# Sprint 4 — Red bayesiana: inferencia y valor esperado (Carta 6)

Notebook de auditoría/ejecución. No duplica lógica: cada celda importa y llama funciones de
`RedBayesiana/codigo_red/*.py`. Sirve como registro reproducible — queda guardado con outputs.

Alcance: cartas 4-6 (verificación retrospectiva de CPD/validación cruzada, cobertura S4/S6/S8,
inferencia C1 y valor esperado). NO calcula RMSE/R² ni compara con LSTM — eso es la carta 7.

In [1]:
import sys
from pathlib import Path

RAIZ = Path.cwd()
while not (RAIZ / "RedBayesiana").exists() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent

CODIGO_RED = RAIZ / "RedBayesiana" / "codigo_red"
sys.path.insert(0, str(CODIGO_RED))

import pandas as pd
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

from ensamblado import ensamblar_conjunto, CLAVE, COLUMNAS_BN, ESTADOS_BN
from red_bayesiana import construir_modelo_manual
from ajuste_cpd import preparar_fold, ajustar_cpd, verificar_cpd, ESS_EVALUADOS, ESS_SELECCIONADO
from validacion_cruzada import ejecutar_validacion_cruzada, FOLDS_ESPERADOS_POR_MATERIA
from valor_esperado import ESTADOS, medias_entrenamiento_por_estado, valor_esperado
from inferencia import ejecutar_inferencia, verificar_resultados, HITOS, RUTA_CSV_PREDICCIONES

sesiones, reg, sem = ensamblar_conjunto()
print(f"sesiones={len(sesiones)}  reg={len(reg)}  sem={len(sem)}")

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/pgmpy/estimators/__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


sesiones=11700  reg=524  sem=6288


## B. Verificación retrospectiva de las Tarjetas 4 y 5

Se reejecutan las mismas funciones ya usadas y reportadas en esas cartas — ningún resultado nuevo, solo se deja constancia ejecutada y guardada en este notebook.

In [2]:
resultados_cv = ejecutar_validacion_cruzada(sesiones)

assert len(resultados_cv) == 14, f"se esperaban 14 pliegues, hay {len(resultados_cv)}"
conteo_por_materia = {}
for r in resultados_cv:
    conteo_por_materia[r["materia"]] = conteo_por_materia.get(r["materia"], 0) + 1
assert conteo_por_materia == FOLDS_ESPERADOS_POR_MATERIA, conteo_por_materia

print("Pliegues por asignatura:", conteo_por_materia, "-> total", len(resultados_cv))

Pliegues por asignatura: {'Algoritmos y Programación': 4, 'Computación Emergente': 4, 'Estructura de Datos': 4, 'Matemáticas Discretas': 2} -> total 14


In [3]:
filas_resumen = []
for r in resultados_cv:
    fila = {
        "materia": r["materia"], "trimestre_prueba": r["trimestre_prueba"],
        "registros_train": r["registros_train"], "registros_test": r["registros_test"],
        "sesiones_train_util": r["sesiones_train_util"], "sesiones_test_util": r["sesiones_test_util"],
        "tema_no_visto_test": r["tema_no_visto_en_test"],
    }
    for ess, v in r["verificacion_cpd_por_ess"].items():
        fila[f"min_prob_ess{ess}"] = v["min_probabilidad_global"]
        fila[f"err_norm_ess{ess}"] = v["max_error_normalizacion"]
    filas_resumen.append(fila)

df_cv = pd.DataFrame(filas_resumen)
assert (df_cv["min_prob_ess1"] > 0).all() and (df_cv["min_prob_ess5"] > 0).all() and (df_cv["min_prob_ess10"] > 0).all()
print("CPD normalizadas y con probabilidad positiva para ESS=1,5,10 en los 14 pliegues.")
df_cv

CPD normalizadas y con probabilidad positiva para ESS=1,5,10 en los 14 pliegues.


,materia,trimestre_prueba,registros_train,registros_test,sesiones_train_util,sesiones_test_util,tema_no_visto_test,min_prob_ess1,err_norm_ess1,min_prob_ess5,err_norm_ess5,min_prob_ess10,err_norm_ess10
0,Algoritmos y Programación,2425-2,87,60,1870,1320,0,0.000003,4.440892e-16,0.000017,4.440892e-16,0.000034,5.551115e-16
1,Algoritmos y Programación,2526-1,120,27,2640,550,0,0.000007,4.440892e-16,0.000035,4.440892e-16,0.000071,3.330669e-16
2,Algoritmos y Programación,2526-2,117,30,2530,660,0,0.000003,4.440892e-16,0.000017,4.440892e-16,0.000034,4.440892e-16
3,Algoritmos y Programación,2526-3,117,30,2530,660,0,0.000003,4.440892e-16,0.000017,4.440892e-16,0.000034,4.440892e-16
4,Computación Emergente,2425-3,109,73,2376,803,38,0.000004,3.330669e-16,0.000020,3.330669e-16,0.000040,4.440892e-16
5,Computación Emergente,2526-1,154,28,2563,616,0,0.000005,3.330669e-16,0.000023,3.330669e-16,0.000046,3.330669e-16
6,Computación Emergente,2526-2,141,41,2299,880,0,0.000002,3.330669e-16,0.000010,3.330669e-16,0.000019,4.440892e-16
7,Computación Emergente,2526-3,142,40,2299,880,40,0.000001,2.220446e-16,0.000006,3.330669e-16,0.000012,4.440892e-16
8,Estructura de Datos,2425-3,78,30,1716,660,0,0.000002,4.440892e-16,0.000012,4.440892e-16,0.000023,2.220446e-16
9,Estructura de Datos,2526-1,78,30,1716,660,0,0.000006,4.440892e-16,0.000029,4.440892e-16,0.000057,3.330669e-16


## C. Cobertura S4/S6/S8 y evidencia parcial

Verificación previa a implementar C1 (ya reportada, reejecutada aquí para trazabilidad): ausencia real de la semana del hito, huecos de numeración semanal, y registros con año académico faltante, que se evalúan con evidencia parcial (Año que cursa omitido, marginalizado por VariableElimination) — nunca imputados ni excluidos.

In [4]:
huecos = 0
sem_sorted = sem.sort_values(list(CLAVE) + ["semana"])
for _, g in sem_sorted.groupby(list(CLAVE)):
    semanas = g["semana"].tolist()
    huecos += sum(1 for i in range(1, len(semanas)) if semanas[i] != semanas[i - 1] + 1)

cobertura = []
for hito in HITOS:
    presentes = set(map(tuple, sem.loc[sem["semana"] == hito, list(CLAVE)].values.tolist()))
    cobertura.append({
        "hito": hito,
        "registros_con_semana": len(presentes),
        "registros_totales": len(reg),
        "sin_semana_hito": len(reg) - len(presentes),
    })

print(f"Huecos de numeración de semana en todo el conjunto: {huecos}")
pd.DataFrame(cobertura)

Huecos de numeración de semana en todo el conjunto: 0


,hito,registros_con_semana,registros_totales,sin_semana_hito
0,4,524,524,0
1,6,524,524,0
2,8,524,524,0


In [5]:
faltan_anio = reg.loc[reg["Año que cursa"].isna(), list(CLAVE)]
print(f"Registros con año académico faltante: {len(faltan_anio)} -- ya NO se excluyen de la "
      "inferencia; se evalúan con evidencia parcial (se omite 'Año que cursa' y "
      "VariableElimination marginaliza esa variable), nunca imputados.")
faltan_anio

Registros con año académico faltante: 3 -- ya NO se excluyen de la inferencia; se evalúan con evidencia parcial (se omite 'Año que cursa' y VariableElimination marginaliza esa variable), nunca imputados.


,estudiante_id,materia,trimestre,seccion
123,anon_259,Computación Emergente,2526-2,1
412,anon_063,Algoritmos y Programación,2526-1,1
414,anon_065,Algoritmos y Programación,2526-1,1


## D. Medias de entrenamiento por estado y pliegue (para el valor esperado)

Calculadas exclusivamente con los registros estudiante-sección de entrenamiento de cada pliegue (`valor_esperado.medias_entrenamiento_por_estado`) — nunca con el trimestre de prueba, nunca con una tabla global.

In [6]:
filas_medias = []
for materia in sorted(sesiones["materia"].unique()):
    trimestres = sorted(sesiones.loc[sesiones["materia"] == materia, "trimestre"].unique())
    for trimestre_prueba in trimestres:
        medias = medias_entrenamiento_por_estado(reg, materia, trimestre_prueba)
        fila = {"materia": materia, "trimestre_prueba": trimestre_prueba}
        fila.update({f"media_{estado}": v for estado, v in medias.items()})
        filas_medias.append(fila)

df_medias = pd.DataFrame(filas_medias)
df_medias

,materia,trimestre_prueba,media_0,media_1-2,media_3-5,media_6-11,media_12 o más
0,Algoritmos y Programación,2425-2,0.0,1.379310,4.263158,8.687500,14.666667
1,Algoritmos y Programación,2526-1,0.0,1.517241,4.074074,8.636364,18.312500
2,Algoritmos y Programación,2526-2,0.0,1.466667,4.000000,8.666667,17.526316
3,Algoritmos y Programación,2526-3,0.0,1.447368,4.105263,8.894737,17.812500
4,Computación Emergente,2425-3,0.0,1.173913,3.785714,8.307692,18.066667
5,Computación Emergente,2526-1,0.0,1.235294,3.826087,7.647059,19.727273
6,Computación Emergente,2526-2,0.0,1.212121,3.916667,7.562500,18.500000
7,Computación Emergente,2526-3,0.0,1.205128,4.000000,7.181818,13.200000
8,Estructura de Datos,2425-3,0.0,1.785714,3.636364,8.166667,20.666667
9,Estructura de Datos,2526-1,0.0,1.666667,3.384615,8.538462,21.500000


## E. Inferencia (Carta 6) — alternativa C1

Evidencia mínima necesaria (los 4 padres directos del objetivo): Año que cursa, Tamaño del grupo, Participaciones de la semana en semana=hito, Participaciones de la semana anterior. Una predicción por registro estudiante-sección y por hito — nunca por sesión.

In [7]:
resultados, resumenes_inferencia = ejecutar_inferencia(sesiones, reg, sem)
print(f"Filas de predicción (registro x hito): {len(resultados)}")
resultados.head()

Filas de predicción (registro x hito): 1572


,materia,trimestre_prueba,estudiante_id,seccion,hito,total_trimestre_real,estado_real,prediccion_continua,posterior_0,posterior_1-2,posterior_3-5,posterior_6-11,posterior_12 o más,ess,n_train_fold,n_test_fold,evidencia_completa,evidencia_anio_que_cursa,evidencia_tamano_grupo,evidencia_participaciones_semana,evidencia_participaciones_semana_anterior
0,Algoritmos y Programación,2425-2,anon_001,1,4,2.0,1-2,2.840077,0.24719,0.351119,0.275279,0.112362,0.01405,5,87,60,True,1,mediano,0,0
1,Algoritmos y Programación,2425-2,anon_001,1,6,2.0,1-2,2.840077,0.24719,0.351119,0.275279,0.112362,0.01405,5,87,60,True,1,mediano,0,0
2,Algoritmos y Programación,2425-2,anon_001,1,8,2.0,1-2,2.840077,0.24719,0.351119,0.275279,0.112362,0.01405,5,87,60,True,1,mediano,0,0
3,Algoritmos y Programación,2425-2,anon_002,1,4,2.0,1-2,2.840077,0.24719,0.351119,0.275279,0.112362,0.01405,5,87,60,True,1,mediano,0,0
4,Algoritmos y Programación,2425-2,anon_002,1,6,2.0,1-2,2.840077,0.24719,0.351119,0.275279,0.112362,0.01405,5,87,60,True,1,mediano,0,0


## F. Verificaciones automáticas

In [8]:
verificacion = verificar_resultados(resultados, resumenes_inferencia, sem)
verificacion

{'n_predicciones_totales': 1572,
 'n_registros_evaluados': 524,
 'n_evidencia_parcial_anio_faltante': 3,
 'n_pliegues': 14}

## G. Tabla resumen de resultados

In [9]:
resumen_por_hito = resultados.groupby("hito")["prediccion_continua"].agg(["count", "mean", "std"])
resumen_por_hito

,count,mean,std
hito,,,
4,524,4.719218,3.938607
6,524,3.975548,3.091772
8,524,4.248274,3.458486


In [10]:
resumen_por_materia_hito = (
    resultados.groupby(["materia", "hito"])
    .agg(
        n=("prediccion_continua", "count"),
        prediccion_media=("prediccion_continua", "mean"),
        real_media=("total_trimestre_real", "mean"),
    )
    .reset_index()
)
resumen_por_materia_hito

,materia,hito,n,prediccion_media,real_media
0,Algoritmos y Programación,4,147,5.885688,5.149660
1,Algoritmos y Programación,6,147,4.280188,5.149660
2,Algoritmos y Programación,8,147,4.868445,5.149660
3,Computación Emergente,4,182,3.515504,3.175824
4,Computación Emergente,6,182,3.125410,3.175824
5,Computación Emergente,8,182,3.419674,3.175824
6,Estructura de Datos,4,108,5.457189,4.907407
7,Estructura de Datos,6,108,4.705336,4.907407
8,Estructura de Datos,8,108,4.213344,4.907407
9,Matemáticas Discretas,4,87,4.350300,4.160920


## H. Archivo CSV generado

Salida completa (una fila por registro estudiante-sección evaluable x hito) para que la Carta 7 la consuma sin repetir la inferencia.

In [11]:
RUTA_CSV_PREDICCIONES.parent.mkdir(parents=True, exist_ok=True)
resultados.to_csv(RUTA_CSV_PREDICCIONES, index=False)
print(f"CSV guardado en: {RUTA_CSV_PREDICCIONES}")
print(f"Filas: {len(resultados)}  Columnas: {list(resultados.columns)}")

CSV guardado en: /Users/nelsoncarrillo/Downloads/tesis-prediccion-participacion/RedBayesiana/resultados/predicciones_bayesiana_s4_s6_s8.csv
Filas: 1572  Columnas: ['materia', 'trimestre_prueba', 'estudiante_id', 'seccion', 'hito', 'total_trimestre_real', 'estado_real', 'prediccion_continua', 'posterior_0', 'posterior_1-2', 'posterior_3-5', 'posterior_6-11', 'posterior_12 o más', 'ess', 'n_train_fold', 'n_test_fold', 'evidencia_completa', 'evidencia_anio_que_cursa', 'evidencia_tamano_grupo', 'evidencia_participaciones_semana', 'evidencia_participaciones_semana_anterior']
